# Hakam — full-corpus feature extraction (Colab GPU)

Produces the **frozen baseline** embeddings for all 8,297 labelled clips.

The diagnostic on 2 Sep 2026 found that frozen Kinetics features encode camera framing
(probe 0.94) but barely encode the foul itself (card probe ~0.47–0.56). So these
embeddings are **not** the final model — they are the baseline that Experiment 1
compares fine-tuning against, which §7 of the brief requires anyway.

### NDA rule, enforced by the structure of this notebook

The dataset is downloaded **into the ephemeral runtime** and dies with it. Only the
embedding cache (~25 MB, no video content) is copied to Drive. Never add a cell that
writes `data/` to Drive.

**Runtime → Change runtime type → GPU** before running anything.

In [ ]:
# 1. Confirm a GPU is actually attached. On CPU this notebook takes ~3 hours
#    instead of ~30 minutes, so fail loudly rather than discovering it later.
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun."
)
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code. If the repo is private, use a personal access token:
#      !git clone https://<TOKEN>@github.com/FerasMad/hakam.git
#    Do not paste a token into a cell you intend to commit.
%cd /content
!git clone https://github.com/FerasMad/hakam.git 2>/dev/null || (cd hakam && git pull)
%cd /content/hakam

In [ ]:
# 3. Dependencies. Colab already ships torch + torchvision built against its CUDA,
#    so install only what is missing rather than letting pip resolve torch again.
!pip install -q transformers SoccerNet opencv-python-headless

import transformers, cv2
print("transformers", transformers.__version__, "| cv2", cv2.__version__)

In [ ]:
# 4. Download the dataset into the runtime. ~3.3 GB, a few minutes on Colab's link.
#
#    The password comes from Colab Secrets, never from a cell. Set it up once:
#      key icon in the left sidebar -> Add new secret
#      name:  SOCCERNET_PASSWORD
#      value: <the NDA password>
#      then toggle notebook access on
#
#    It lives in your Google account: never in the notebook, never in git, never in
#    a cell output. getpass is only the fallback if the secret is missing.
from SoccerNet.Downloader import SoccerNetDownloader

try:
    from google.colab import userdata

    password = userdata.get("SOCCERNET_PASSWORD")
except Exception:
    from getpass import getpass

    password = getpass("SOCCERNET_PASSWORD secret not found - enter password: ")

downloader = SoccerNetDownloader(LocalDirectory="/content/hakam/data/mvfouls")
downloader.password = password
downloader.downloadDataTask(task="mvfouls", split=["train", "valid", "test"])

In [ ]:
# 5. Normalise the layout.
#
#    Two traps here. The zips extract FLAT - every split writes action_0,
#    action_1, ... so if they land in one directory the splits silently
#    overwrite each other. And the downloader's own layout varies by version:
#    it may nest under mvfouls/mvfouls/, may pre-extract, and the archives may
#    be encrypted with the same NDA password.
#
#    So this cell discovers the real layout instead of assuming one, and says
#    out loud what it found. An earlier version guessed at the zip names, found
#    nothing, printed nothing useful, and left cell 6 to fail confusingly.
from pathlib import Path
import shutil
import zipfile

root = Path("/content/hakam/data/mvfouls")

print("--- what the downloader actually produced ---")
entries = sorted(root.rglob("*"))
for p in entries[:40]:
    detail = f"  ({p.stat().st_size / 1e6:.1f} MB)" if p.is_file() else "/"
    print(f"  {p.relative_to(root)}{detail}")
if len(entries) > 40:
    print(f"  ... and {len(entries) - 40} more")
print()

for proper, lower in {"Train": "train", "Valid": "valid", "Test": "test"}.items():
    target = root / proper
    if (target / "annotations.json").exists():
        print(f"{proper}: already in place")
        continue

    # Any archive under root belonging to this split. Exact stem wins, so
    # "test.zip" is never shadowed by something merely containing "test".
    archive = (
        next((z for z in root.rglob("*.zip") if z.stem.lower() == lower), None)
        or next((z for z in root.rglob("*.zip") if lower in z.stem.lower()), None)
    )

    if archive is not None:
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zf:
            try:
                zf.extractall(target)
            except RuntimeError:
                # Encrypted archive - same NDA password, taken from the secret
                # read in cell 4 so it never appears in this notebook.
                zf.extractall(target, pwd=password.encode())
        print(f"{proper}: extracted {archive.name}")
        continue

    # Or the downloader already extracted it somewhere else in the tree.
    found = next(
        (a.parent for a in root.rglob("annotations.json")
         if proper.lower() in str(a.parent).lower()),
        None,
    )
    if found is not None and found != target:
        shutil.move(str(found), str(target))
        print(f"{proper}: moved {found} -> {target}")
    else:
        print(f"{proper}: NOT FOUND - check the layout printed above")

print()
for proper in ["Train", "Valid", "Test"]:
    d = root / proper
    print(f"{proper:6} annotations.json={(d / 'annotations.json').exists()}  "
          f"actions={len(list(d.glob('action_*')))}")

In [ ]:
# 6. Verify the splits match the official counts before spending GPU time.
#
#    Reports every split before raising, so one bad split does not hide the
#    state of the other two.
import sys

sys.path.insert(0, "/content/hakam")

from pathlib import Path

from src import config
from src.data.annotations import load_split, verify_clips_exist

problems = []
for split in ["train", "valid", "test"]:
    split_dir = config.DATA_ROOT / "mvfouls" / config.SPLIT_DIRS[split]
    if not (split_dir / "annotations.json").exists():
        print(f"{split:6} MISSING annotations.json at {split_dir}")
        problems.append(split)
        continue

    actions, clips = load_split(split_dir)
    missing = verify_clips_exist(clips)
    official = config.SPLIT_SIZES[split]
    flag = "" if len(actions) == official and missing.empty else "   <-- WRONG"
    print(f"{split:6} actions={len(actions):>5} (official {official:>4})  "
          f"clips={len(clips):>5}  missing={len(missing)}{flag}")
    if len(actions) != official or not missing.empty:
        problems.append(split)

if problems:
    raise SystemExit(
        f"splits not ready: {', '.join(problems)}. Re-run cell 5 and read the "
        "layout it prints - the usual cause is the archives landing somewhere "
        "the extractor did not look, or the splits colliding because the zips "
        "extract flat."
    )
print("\nall splits verified - safe to extract")

In [ ]:
# 7. Extract. Both backbones in one go - the second costs only minutes while the
#    runtime is warm, and it pre-pays Experiment 1's backbone comparison.
#
#    Window is 43-107 (2.56 s), reproducing VideoMAE's 16-frames-at-stride-4
#    pretraining. The published 63-87 window spans 0.96 s and yields near-duplicate
#    frames; it lost on every measurement.
!python scripts/extract_all.py \
    --backbones videomae_small videomae_base \
    --splits train valid test \
    --batch-size 32

In [ ]:
# 8. Copy ONLY the embedding cache to Drive. No video, ever.
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/hakam/features_cache")
dest.mkdir(parents=True, exist_ok=True)

for f in sorted(Path("/content/hakam/features_cache").glob("*")):
    shutil.copy2(f, dest / f.name)
    print(f"{f.name}  {f.stat().st_size / 1e6:.1f} MB")

print(f"\ncopied to {dest}")

## Next

Download `features_cache/` from Drive into the local repo, then train the cascade heads
(Cycle 1). Experiments come in Cycle 2 — fine-tuning is the one that matters, since the
diagnostic showed frozen features carry only weak signal about the foul itself.

Fine-tuning trains the backbone, so it needs decoded **frames**, not this cache. It runs
in its own notebook.